# Round 3 — arm 1: the SCAN PROFILE (Lever 7)

**One arm, one run.** This notebook trains the scan-profile arm. Its control is already on disk — `data/checkpoints/r3-tupnew-stage2-best`, the tuplet A/B's `tupnew` arm — because that checkpoint used **this corpus, this split, this recipe and this seed**, and the only thing this notebook changes is the augmentation mix:

| | screenshot | photo | scan |
|---|---|---|---|
| control (`r3-tupnew-stage2-best`) | 0.65 | 0.35 | — |
| **this arm** | **0.55** | **0.20** | **0.25** |

The scan share comes mostly out of `PHOTO_SHARE`, whose own 0.35 was a guess from "uploads are mostly screenshots". **The mix is pre-registered — see `docs/rung3/levers.md` Lever 7 — and is not to be tuned after seeing a result.**

**Why the arm exists.** `src/vision/augment.py` had exactly two profiles, and a flatbed scan of a TRT-era print is neither: flat lighting and no perspective, but speckle, broken thin lines, ink spread, bleed-through and threshold damage. **93% of exam pages are scans.** Its own comment has said *"Revisit against real usage at Rung 3"* since July.

⚠ **The trade is accepted knowingly**: aiming augmentation at scans optimises the *exam*, and we do not know that the exam's medium is what the app's users upload (n = 2). That is what the no-regression clause on the born-digital / easy side is for.

⚠ **The corpus is `strips_v5_tupnew` and that is deliberate** — the same corpus as the control. If you find yourself looking for a "scan corpus", there isn't one and there must not be: a second variable is what made Round 3 unattributable twice already.

⚠ **Exam strips are not on this VM and the exam is not read here.** It is read once, later, on Round 3's final model.

⚠ **The read happens on the Mac**, not here — `scripts/rung3/split_realval_tiers.py` builds the scoring pools and the control's numbers are already recorded in `docs/METRICS-DIAGNOSTICS.md`.

In [ ]:
# ===== THE ONLY KNOBS IN THIS NOTEBOOK — and they are pre-registered, not free =====
ARM = 'scan'
PHOTO_SHARE = 0.20
SCAN_SHARE = 0.25
STRIPS = 'data/synthetic/strips_v5_tupnew'   # SAME corpus as the control. Do not change.
ZIP = 'tnc_round3_scan_colab.zip'
DRIVE = '/content/drive/MyDrive/tnc'
print(ARM, STRIPS, ZIP, f'screenshot {1-PHOTO_SHARE-SCAN_SHARE:.2f} / photo {PHOTO_SHARE} / scan {SCAN_SHARE}')

In [ ]:
# Which GPU did we get? (T4 16GB / L4 24GB / A100 40GB)
!nvidia-smi

In [ ]:
# Mount Google Drive (approve the popup).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%time
# Copy the package Drive -> VM disk and unzip (fast local disk for the dataloader).
!cp {DRIVE}/{ZIP} /content/
!rm -rf /content/tnc && mkdir /content/tnc
!cd /content/tnc && unzip -q /content/{ZIP}

# WHICH CORPUS IS ACTUALLY ON DISK. This arm ships tupnew, so the assertion is the mirror of the
# tuplet A/B's: legacyTupletMark must be False. A tupctl upload here would train a different
# corpus AND a different mix, and nothing later could separate the two.
import json
cfg = json.load(open(f'/content/tnc/{STRIPS}/render_config.json'))
print(cfg)
assert cfg['legacyTupletMark'] is False, f'{ZIP} is not the tupnew corpus — the control is'
assert cfg['thinSharps'] is True and cfg['printNoise'] is False
!wc -l /content/tnc/{STRIPS}/manifest.jsonl
!python -c "import json;s=json.load(open('/content/tnc/data/split_v4.json'));print('train',len(s['train_pieces']),'val',len(s['val_pieces']))"

In [ ]:
# Dependencies (torch + torchvision are preinstalled on Colab).
!pip -q install transformers albumentations opencv-python-headless

In [ ]:
# ===== THE MIX IS THE EXPERIMENT — prove it is on before spending a GPU hour =====
# Nothing downstream records the augmentation mix: corpus, split and checkpoint are identical
# between this arm and its control. train.py prints the mix in its own startup line, and this cell
# is the assertion that the number reaching the Augmenter is the pre-registered one.
%cd /content/tnc
import sys
sys.path.insert(0, 'src/vision')
from augment import Augmenter
a = Augmenter(seed=7, photo_share=PHOTO_SHARE, scan_share=SCAN_SHARE)
assert (a.photo_share, a.scan_share) == (0.20, 0.25), (a.photo_share, a.scan_share)
import numpy as np
draws = [('photo' if r < a.photo_share else 'scan' if r < a.photo_share + a.scan_share
          else 'screenshot') for r in (a.rng.random() for _ in range(20000))]
from collections import Counter
print({k: round(v / 200, 1) for k, v in Counter(draws).items()}, '% over 20k draws')
# and one scan-profile image, so a broken op is caught here rather than at step 6000
img = np.full((336, 900, 3), 255, np.uint8); img[150:160, ::7] = 0
out = a(img.copy(), profile='scan')
print('scan profile ok:', out.shape, out.dtype, 'changed:', not np.array_equal(out, img))

In [ ]:
# SHAKEOUT (~3 min): 150 tiny steps from BASE — a WIRING smoke, not a result.
# Expect: `vocab: +25 tokens -> 100 ids`, the three real pools listed, `exam-disjointness OK`,
# `augment=on (screenshot 0.55 / photo 0.20 / scan 0.25)`, and val loss FALLING.
%cd /content/tnc
!python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir data/real/rung3/strips_nota --real-dir data/real/rung3/strips_r1 \
    --real-dir data/real/rung3/strips_tup \
    --photo-share {PHOTO_SHARE} --scan-share {SCAN_SHARE} \
    --every-share 0.15 --out-dir /content/r3-shakeout \
    --lr 3e-5 --warmup-steps 30 --max-steps 150 --batch-size 8 \
    --limit-val 40 --eval-every 50 --save-every 50 --log-every 25 --num-workers 2

In [ ]:
# ===== CALIBRATE THROUGHPUT ON *THIS* RUNTIME (~2-3 min) — before any long run =====
#   hours = (steps * batch) / samples_per_sec / 3600
# ⚠ The scan profile is CPU work per sample (speckle, dropout, threshold), and Round-1's T4 run was
# already augmentation-CPU-bound rather than GPU-bound. If this reads slower than the tupnew run
# did, that is the profile, not a broken runtime — raise --num-workers before raising anything else.
# ⚠ Whatever you set, the STEP COUNTS AND BATCH SIZE must match the control's: 6000 @ 16, then
# 2000 @ 16. --num-workers and the GPU model do not change the result; those two do.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!nproc
%cd /content/tnc
!python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir data/real/rung3/strips_nota --real-dir data/real/rung3/strips_r1 \
    --real-dir data/real/rung3/strips_tup \
    --photo-share {PHOTO_SHARE} --scan-share {SCAN_SHARE} \
    --every-share 0.15 --out-dir /content/calib \
    --lr 3e-5 --warmup-steps 20 --max-steps 60 --batch-size 16 \
    --limit-val 8 --eval-every 60 --save-every 60 --log-every 20 --num-workers 10

In [ ]:
# ===== STAGE 1 — carry-dominant SYNTHETIC ONLY, from BASE =====
# No --real-dir: this builds the carry-native synthetic checkpoint stage 2 specialises. It is also
# where the scan profile does most of its work, because real strips train CLEAN (--augment-real is
# off, and stays off — double-degrading a blurry nota scan buries its signal).
# ⚠ `-u` is not cosmetic: Colab block-buffers a subprocess's stdout, so without it the log
# lines sit in an 8 KB buffer and a healthy run looks frozen for minutes at a time.
# Recipe identical to the control's, except the mix.
%cd /content/tnc
!python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --photo-share {PHOTO_SHARE} --scan-share {SCAN_SHARE} \
    --every-share 0.15 --out-dir {DRIVE}/r3-{ARM}-stage1 \
    --lr 3e-5 --max-steps 6000 --batch-size 16 --num-workers 10  # ~nproc-2; T4 (2 vCPU) use 2

In [ ]:
# ===== STAGE 2 — real-SPECIALISATION fine-tune from stage 1 =====
# Fresh LOW lr + short warmup from the stage-1 checkpoint. `:9` as in the control — the suffix
# oversamples each real pool so real is ~1/3 of batches. It must be THE SAME AS THE CONTROL'S.
#
# Selection caveat carried over: oversampled real overfits fast and `best` is picked on a
# synth-dominated val mix — so both `best` and `last` come home. ⚠ The control on disk is
# `r3-tupnew-stage2-BEST`, so `best` is the comparison and `last` is reported beside it.
%cd /content/tnc
!python -u src/vision/train.py --model {DRIVE}/r3-{ARM}-stage1/best \
    --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir data/real/rung3/strips_nota:9 \
    --real-dir data/real/rung3/strips_r1:9 \
    --real-dir data/real/rung3/strips_tup:9 \
    --photo-share {PHOTO_SHARE} --scan-share {SCAN_SHARE} \
    --every-share 0.15 --out-dir {DRIVE}/r3-{ARM}-stage2 \
    --lr 1e-5 --warmup-steps 100 --max-steps 2000 --batch-size 16 --num-workers 10

In [ ]:
# RESUME after a disconnect: re-run the setup cells, then this with the SAME flags as the stage
# you were running (edit out-dir/flags to match). --resume reloads model+optimizer+scheduler from
# <out-dir>/last and ignores --model.
%cd /content/tnc
!python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --photo-share {PHOTO_SHARE} --scan-share {SCAN_SHARE} \
    --every-share 0.15 --out-dir {DRIVE}/r3-{ARM}-stage1 \
    --lr 3e-5 --max-steps 6000 --batch-size 16 --num-workers 10 --resume

In [ ]:
# ===== SANITY ONLY — did anything break? =====
# NOT the pre-registered number. That is read on the Mac, on the tier/medium pools, paired against
# the control. This cell exists so a broken run is caught before it is downloaded, and to see
# `best` and `last` side by side.
%cd /content/tnc
!python src/vision/make_realval_pool.py --real-dir data/real/rung3/strips_nota \
    --real-dir data/real/rung3/strips_r1 --real-dir data/real/rung3/strips_tup \
    --split data/split_v4.json

for ck in [f'r3-{ARM}-stage2/best', f'r3-{ARM}-stage2/last']:
    print('=' * 70, '\n==', ck)
    !python src/vision/eval_omr.py --checkpoint {DRIVE}/{ck} \
        --strips-dir data/real/rung3/_realval --split none --show-errors 0

## After the run

1. **Download the stage-2 checkpoints** from `MyDrive/tnc/r3-scan-stage2/` into `data/checkpoints/` on the Mac (`best` is the comparison, `last` comes too).
2. **Build the scoring pools** (idempotent; already built once):
   ```bash
   .venv-ml/bin/python scripts/rung3/split_realval_tiers.py --force
   ```
3. **Read the pre-registered numbers**, arm and control, on the same pools:
   ```bash
   for p in _hard _easy _mid _scan _borndigital ""; do
     .venv-ml/bin/python src/vision/eval_omr.py --checkpoint data/checkpoints/<arm-ckpt> \
         --strips-dir data/real/rung3/_realval_v2$p --split none --show-errors 0
   done
   ```
   The control's numbers are already recorded — do not re-derive them from memory.
4. **Apply the decision rule in `docs/rung3/levers.md` Lever 7.** It is written down before the run; do not re-derive it from the result. A null is reported as a null, and the mix is not re-tuned to chase a win.
5. ⚠ **Two limits to quote beside whatever comes back**: the pools are 47–202 strips, so only a large move is resolvable; and `edits/page` on them is edits per *page fragment* (2.1–2.7 strips a page), not per page.
6. **The exam is still unread.** It is one shot, on Round 3's final model, against the floors in `docs/rung3/round3-criteria.md`.